### Load and clean the session data

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

with open("../data/acndata_sessions.json", "r") as f:
    data = json.load(f)

sessions = pd.json_normalize(data["_items"])

sessions = sessions[
    [
        "sessionID",
        "stationID",
        "spaceID",
        "connectionTime",
        "disconnectTime",
        "doneChargingTime",
        "kWhDelivered"
    ]
].copy()

sessions["connectionTime"] = pd.to_datetime(sessions["connectionTime"], errors="coerce", utc=True)
sessions["disconnectTime"] = pd.to_datetime(sessions["disconnectTime"], errors="coerce", utc=True)
sessions["doneChargingTime"] = pd.to_datetime(sessions["doneChargingTime"], errors="coerce", utc=True)

sessions = sessions.dropna(subset=["stationID", "connectionTime", "disconnectTime"])

sessions = sessions[sessions["disconnectTime"] > sessions["connectionTime"]]

sessions.head()


,sessionID,stationID,spaceID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered
0,1_1_193_829_2019-01-02 01:00:51.413435,1-1-193-829,AG-1F03,2019-01-02 01:00:51+00:00,2019-01-02 02:39:46+00:00,2019-01-02 02:39:37+00:00,10.143
1,1_1_191_789_2019-01-02 13:39:11.359003,1-1-191-789,AG-4F52,2019-01-02 13:39:11+00:00,2019-01-03 01:19:57+00:00,2019-01-02 15:37:12+00:00,5.871
2,1_1_178_823_2019-01-02 13:44:26.828039,1-1-178-823,AG-1F08,2019-01-02 13:44:27+00:00,2019-01-02 22:37:33+00:00,2019-01-02 19:18:16+00:00,12.094
3,1_1_193_829_2019-01-02 13:47:38.465648,1-1-193-829,AG-1F03,2019-01-02 13:47:38+00:00,2019-01-02 19:01:31+00:00,2019-01-02 15:06:07+00:00,2.425
4,1_1_193_819_2019-01-02 13:53:40.716472,1-1-193-819,AG-1F06,2019-01-02 13:53:41+00:00,2019-01-02 21:40:03+00:00,2019-01-02 16:45:46+00:00,14.331


### Convert sessions into station-time data

In [4]:
import math


def build_station_usage_panel(sessions, freq="15min", horizon_minutes=60):
    step = pd.Timedelta(freq)
    horizon_steps = max(1, math.ceil(pd.Timedelta(minutes=horizon_minutes) / step))

    start_time = sessions["connectionTime"].min().floor(freq)
    end_time = sessions["disconnectTime"].max().ceil(freq)

    time_index = pd.date_range(start=start_time, end=end_time, freq=freq)

    all_station_panels = []

    for station_id, station_sessions in sessions.groupby("stationID"):
        station_sessions = station_sessions.copy()

        start_bins = station_sessions["connectionTime"].dt.floor(freq)
        end_bins = station_sessions["disconnectTime"].dt.ceil(freq)

        starts = (
            start_bins
            .value_counts()
            .reindex(time_index, fill_value=0)
            .sort_index()
        )

        ends = (
            end_bins
            .value_counts()
            .reindex(time_index, fill_value=0)
            .sort_index()
        )

        active_sessions = (starts - ends).cumsum().clip(lower=0)

        occupied_now = (active_sessions > 0).astype(int)

        kwh_started_now = (
            station_sessions
            .groupby(start_bins)["kWhDelivered"]
            .sum()
            .reindex(time_index, fill_value=0)
            .sort_index()
        )

        station_panel = pd.DataFrame({
            "timestamp": time_index,
            "stationID": station_id,
            "occupied_now": occupied_now.values,
            "sessions_started_now": starts.values,
            "kwh_started_now": kwh_started_now.values
        })

        future_occupied = station_panel["occupied_now"].shift(-1)

        station_panel["will_be_used_next_horizon"] = (
            future_occupied
            .iloc[::-1]
            .rolling(window=horizon_steps, min_periods=horizon_steps)
            .max()
            .iloc[::-1]
        )

        all_station_panels.append(station_panel)

    panel = pd.concat(all_station_panels, ignore_index=True)

    return panel

### Create useful prediction features

In [6]:
horizon_minutes = 60
freq = "15min"

panel = build_station_usage_panel(
    sessions=sessions,
    freq=freq,
    horizon_minutes=horizon_minutes
)

panel.head()

,timestamp,stationID,occupied_now,sessions_started_now,kwh_started_now,will_be_used_next_horizon
0,2019-01-02 01:00:00+00:00,1-1-178-817,0,0,0.0,0.0
1,2019-01-02 01:15:00+00:00,1-1-178-817,0,0,0.0,0.0
2,2019-01-02 01:30:00+00:00,1-1-178-817,0,0,0.0,0.0
3,2019-01-02 01:45:00+00:00,1-1-178-817,0,0,0.0,0.0
4,2019-01-02 02:00:00+00:00,1-1-178-817,0,0,0.0,0.0


In [7]:
panel["hour"] = panel["timestamp"].dt.hour
panel["dayofweek"] = panel["timestamp"].dt.dayofweek
panel["month"] = panel["timestamp"].dt.month
panel["is_weekend"] = panel["dayofweek"].isin([5, 6]).astype(int)

panel = panel.sort_values(["stationID", "timestamp"])

panel["past_occupied_rate_1h"] = (
    panel
    .groupby("stationID")["occupied_now"]
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
)

panel["past_occupied_rate_3h"] = (
    panel
    .groupby("stationID")["occupied_now"]
    .transform(lambda x: x.shift(1).rolling(12, min_periods=1).mean())
)

panel["past_sessions_24h"] = (
    panel
    .groupby("stationID")["sessions_started_now"]
    .transform(lambda x: x.shift(1).rolling(96, min_periods=1).sum())
)

panel["past_kwh_24h"] = (
    panel
    .groupby("stationID")["kwh_started_now"]
    .transform(lambda x: x.shift(1).rolling(96, min_periods=1).sum())
)

panel = panel.fillna(0)

panel.head()

,timestamp,stationID,occupied_now,sessions_started_now,kwh_started_now,will_be_used_next_horizon,hour,dayofweek,month,is_weekend,past_occupied_rate_1h,past_occupied_rate_3h,past_sessions_24h,past_kwh_24h
0,2019-01-02 01:00:00+00:00,1-1-178-817,0,0,0.0,0.0,1,2,1,0,0.0,0.0,0.0,0.0
1,2019-01-02 01:15:00+00:00,1-1-178-817,0,0,0.0,0.0,1,2,1,0,0.0,0.0,0.0,0.0
2,2019-01-02 01:30:00+00:00,1-1-178-817,0,0,0.0,0.0,1,2,1,0,0.0,0.0,0.0,0.0
3,2019-01-02 01:45:00+00:00,1-1-178-817,0,0,0.0,0.0,1,2,1,0,0.0,0.0,0.0,0.0
4,2019-01-02 02:00:00+00:00,1-1-178-817,0,0,0.0,0.0,2,2,1,0,0.0,0.0,0.0,0.0


### Train a model to predict future station usage

In [8]:
target = "will_be_used_next_horizon"

model_df = panel.dropna(subset=[target]).copy()

features = [
    "stationID",
    "occupied_now",
    "hour",
    "dayofweek",
    "month",
    "is_weekend",
    "past_occupied_rate_1h",
    "past_occupied_rate_3h",
    "past_sessions_24h",
    "past_kwh_24h"
]

X = model_df[features]
y = model_df[target].astype(int)

model_df = model_df.sort_values("timestamp")

split_index = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_index]
test_df = model_df.iloc[split_index:]

X_train = train_df[features]
y_train = train_df[target].astype(int)

X_test = test_df[features]
y_test = test_df[target].astype(int)

#Train a model
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

numeric_features = [
    "occupied_now",
    "hour",
    "dayofweek",
    "month",
    "is_weekend",
    "past_occupied_rate_1h",
    "past_occupied_rate_3h",
    "past_sessions_24h",
    "past_kwh_24h"
]

categorical_features = ["stationID"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            min_samples_leaf=5
        ))
    ]
)

model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Evaluate the model

In [9]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))

confusion_matrix(y_test, y_pred)

Accuracy: 0.983
Precision: 0.956
Recall: 1.0
F1: 0.977
ROC AUC: 0.996


array([[1287,   35],
       [   0,  758]])

### Save predictions for Tableau

In [10]:
predictions = test_df[
    [
        "timestamp",
        "stationID",
        "occupied_now",
        "will_be_used_next_horizon"
    ]
].copy()

predictions["predicted_usage"] = y_pred
predictions["predicted_usage_probability"] = y_prob

predictions.head()

,timestamp,stationID,occupied_now,will_be_used_next_horizon,predicted_usage,predicted_usage_probability
2360,2019-01-03 17:00:00+00:00,1-1-179-791,1,1.0,1,0.995635
9360,2019-01-03 17:00:00+00:00,1-1-193-827,1,1.0,1,0.974709
4560,2019-01-03 17:00:00+00:00,1-1-179-815,0,0.0,0,0.193567
160,2019-01-03 17:00:00+00:00,1-1-178-817,1,1.0,1,0.980743
1160,2019-01-03 17:00:00+00:00,1-1-179-779,1,1.0,1,0.976829


In [11]:
Path("../outputs").mkdir(exist_ok=True)

predictions.to_csv("../outputs/station_usage_predictions.csv", index=False)
panel.to_csv("../outputs/station_usage_panel.csv", index=False)